# Assessment 1 — Historical AML analysis with Apache Spark

**Student ID:** 33602190  
**Status:** Step 1 — environment setup only. Phase 1 approval confirmed by the student.

This notebook will compare equivalent DataFrame and Spark SQL analyses of the IBM HI-Small AML dataset, followed by system-performance experiments. No AML findings have been produced yet.

## 1. Setup and scope

Use the project `.venv` Python kernel (see README). Run cells from top to bottom. This milestone verifies that Python can start Spark and execute a small calculation. Dataset loading, query design and performance measurements belong to later milestones.

The full dataset versus subset decision must follow the approved proposal and any tutor guidance. No subset is selected here.

**AI assistance:** OpenAI Codex assisted with this environment setup and notebook scaffold. The student must review and understand the code and maintain an acknowledgement as work progresses.

### 1.1 Configure Python and Java

Spark uses a Java process. The code respects an existing `JAVA_HOME`; otherwise it locates the project-local Java 17 runtime, Homebrew Java 17 on macOS, or Java from PATH. `PYSPARK_PYTHON` makes Python workers use the same environment as this notebook. Loopback networking keeps the local Spark driver on this computer.

Restart the kernel before changing memory settings: they must be applied before the Java process starts.

In [1]:
import os
import platform
import shutil
import subprocess
import sys
from pathlib import Path

if not os.environ.get("JAVA_HOME"):
    java_candidates = sorted(Path(".runtime").glob("jdk-17*/Contents/Home")) + [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
    ]
    for candidate in java_candidates:
        if (candidate / "bin" / "java").exists():
            os.environ["JAVA_HOME"] = str(candidate)
            break

java_home = os.environ.get("JAVA_HOME")
java_executable = str(Path(java_home) / "bin" / "java") if java_home else shutil.which("java")
if not java_executable:
    raise RuntimeError("Install Java 17 and set JAVA_HOME; see README.")
java_check = subprocess.run([java_executable, "-version"], capture_output=True, text=True)
if java_check.returncode != 0:
    raise RuntimeError(java_check.stderr)

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"

print("Python:", platform.python_version())
print("Operating system:", platform.platform())
print("Architecture:", platform.machine())
print("Available logical CPUs:", os.cpu_count())
print("Java:", (java_check.stderr or java_check.stdout).splitlines()[0])

Python: 3.11.17
Operating system: macOS-26.2-arm64-arm-64bit
Architecture: arm64
Available logical CPUs: 14
Java: openjdk version "17.0.20.1" 2026-08-18


### 1.2 Start a local Spark session

`local[4]` allows up to four task threads on this computer; it is not a four-machine cluster. The driver heap is set to 4 GB. Local execution shares the driver JVM rather than allocating separate remote executor heaps.

Eight shuffle partitions are an initial setup value, not an experimentally justified optimum. Partition selection and performance analysis will be addressed in Part B. UTC makes timestamp handling reproducible; it does not establish the source dataset's real-world timezone.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("Assessment1-AML")
    .master("local[4]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.adaptive.enabled", "true")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

settings = {
    "Spark version": spark.version,
    "Execution mode": spark.sparkContext.master,
    "Default parallelism": spark.sparkContext.defaultParallelism,
    "Driver memory configuration": spark.sparkContext.getConf().get("spark.driver.memory"),
    "Actual JVM maximum heap (GiB)": round(spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3, 2),
    "Executor memory configuration": spark.sparkContext.getConf().get("spark.executor.memory"),
    "Separate executor heaps": "Not applicable in local mode",
    "Shuffle partitions": spark.conf.get("spark.sql.shuffle.partitions"),
    "Adaptive query execution": spark.conf.get("spark.sql.adaptive.enabled"),
    "Session timezone": spark.conf.get("spark.sql.session.timeZone"),
}
for name, value in settings.items():
    print(f"{name}: {value}")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/03 08:35:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 3.5.7
Execution mode: local[4]
Default parallelism: 4
Driver memory configuration: 4g
Actual JVM maximum heap (GiB): 4.0
Executor memory configuration: 4g
Separate executor heaps: Not applicable in local mode
Shuffle partitions: 8
Adaptive query execution: true
Session timezone: UTC


### 1.3 Verify execution with a small calculation

The following action executes a Spark aggregation on integers 0–9. The expected count is 10 and the expected sum is 45. This is only an environment check, not the assessment business query or a performance benchmark.

In [3]:
setup_result = (
    spark.range(10)
    .agg(F.count("id").alias("row_count"), F.sum("id").alias("id_sum"))
    .first()
)
assert setup_result["row_count"] == 10
assert setup_result["id_sum"] == 45
print(f"PASS — count: {setup_result['row_count']}; sum: {setup_result['id_sum']}")

PASS — count: 10; sum: 45


### 1.4 Check local input locations

Only file presence is checked in Step 1. CSV contents and schemas will be validated in Step 2. Start the notebook from the repository root.

In [4]:
project_directory = Path.cwd()
data_directory = project_directory / "data"
for filename in ("HI-Small_Trans.csv", "HI-Small_accounts.csv"):
    input_file = data_directory / filename
    if not input_file.is_file():
        raise FileNotFoundError(f"Missing data/{filename}; follow data/README.md")
    print(f"Found data/{filename} ({input_file.stat().st_size:,} bytes)")

Found data/HI-Small_Trans.csv (475,664,283 bytes)
Found data/HI-Small_accounts.csv (34,053,187 bytes)


### 1.5 Release resources

Stop Spark after this setup check. Later milestones will place analysis cells before cleanup. The Spark Web UI is only available while the session is running; capture Part B evidence before stopping it.

In [5]:
spark.stop()
print("Spark session stopped cleanly.")

Spark session stopped cleanly.


## Next milestone

Step 2 will load both CSVs with explicit schemas, assign distinct sender and receiver account names, and validate the input data. No analytical conclusions or performance comparisons are claimed in this setup milestone.

## Technical reference

[Apache Spark 3.5.7 installation requirements](https://spark.apache.org/docs/3.5.7/api/python/getting_started/install.html) describe supported Python and Java versions.